# Infrastructure placement — results: south-bank levee, dz=5m
**Model:** `ahr_river_v03_Marg_additionalsrc_velocity_100m_cutpolygon_warmstart_infra_southonly_dz5m`

## Background
Part of the strategic hydraulic-infrastructure placement study for the Bad Neuenahr-Ahrweiler /
Heimersheim reach (see `setup_infrastructure_placement_runs.py`). This scenario keeps only the
**south-bank** levee segment (the north bank is left unprotected), with crest elevation set to
the local ground level + 5 m (`dz=5`).

All other settings (grid, discharge forcing, warm-start spinup restart) are identical to the
baseline run `ahr_river_v03_Marg_additionalsrc_velocity_100m_cutpolygon_warmstart` — the only
difference is the added `sfincs.weir` file. That baseline is used below as the
"no-infrastructure" reference for evaluating the levee's effect.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

project_root  = Path.cwd().parent  # repository root (run notebooks from notebooks/)
model_root    = project_root / 'models' / 'ahr_river_v03_Marg_additionalsrc_velocity_100m_cutpolygon_warmstart_infra_southonly_dz5m'
baseline_root = project_root / 'models' / 'ahr_river_v03_Marg_additionalsrc_velocity_100m_cutpolygon_warmstart'
obs_dir       = project_root / 'data' / 'observations' / 'w'
mapping_file  = project_root / 'data' / 'observations' / 'mapping.csv'
tref          = pd.Timestamp('2021-07-12 00:00:00')
hmin          = 0.05   # minimum depth to consider wet [m]

# Evaluation window for the infrastructure study (Bad Neuenahr-Ahrweiler / Heimersheim), EPSG:32632.
# Sized from the ACTUAL simulated flood extent (baseline peak depth > hmin) in this reach, not just
# the digitized levee lines' own bbox -- the levee-line bbox alone (y up to 5600760) clips off a real,
# fairly dense band of flooding that extends north to about y=5601450 (tapering from a peak density
# around y=5600750-5600950). Checked by binning wet-cell counts in 100m latitude bands.
WIN_X = (362000.0, 371200.0)
WIN_Y = (5598800.0, 5602100.0)

print('model_root exists:', model_root.exists())
print('Files:', [f.name for f in model_root.iterdir() if f.suffix in ('.nc', '.rst', '.log', '.weir')])

## 1. Check t=0 water depth (warm-start sanity check)
Same warm-start restart as the baseline event, so water should already be present in the channels
at the very first timestep, same as every other run derived from this source event.

In [ ]:
ds_map = xr.open_dataset(model_root / 'sfincs_map.nc', decode_times=False, mask_and_scale=True)

time_sec = ds_map['time'].values.astype(float)
map_times = pd.to_datetime(time_sec, unit='s', origin=tref)
ds_map = ds_map.assign_coords(time=map_times)

print('Map time range:', map_times[0], '->', map_times[-1])
print('Number of timesteps:', len(map_times))

zs = ds_map['zs']
zb = ds_map['zb']
da_h = (zs - zb).where(zb > -9999)

h0 = da_h.isel(time=0).values
wet_at_t0 = np.nansum(h0 > hmin)
print(f'\nWet cells at t=0 (depth > {hmin} m): {wet_at_t0}')
print(f'Max depth at t=0: {np.nanmax(h0):.3f} m')

x0_m, y0_m, dx, dy = 344900.0, 5581800.0, 100.0, 100.0
n_rows, n_cols = h0.shape
x_1d = x0_m + (np.arange(n_cols) + 0.5) * dx
y_1d = y0_m + (np.arange(n_rows) + 0.5) * dy
X, Y = np.meshgrid(x_1d, y_1d)

fig, ax = plt.subplots(figsize=(9, 7))
h0_plot = np.where(h0 > hmin, h0, np.nan)
im = ax.pcolormesh(X, Y, h0_plot, cmap='Blues', vmin=0, vmax=3, shading='nearest')
plt.colorbar(im, ax=ax, label='Water depth [m]')
ax.set_title(f'Water depth at t=0 ({map_times[0].strftime("%Y-%m-%d %H:%M")}) - southonly dz5m')
ax.set_xlabel('Easting [m]'); ax.set_ylabel('Northing [m]')
ax.set_aspect('equal')
plt.tight_layout(); plt.show()

## 2. Gauge station time series — this scenario vs baseline (no infrastructure) vs observed
Altenahr sits upstream of the protected reach, Bad Bodendorf just downstream — neither is inside
the levee corridor itself, but comparing them shows whether the south-bank levee measurably
changes water levels bracketing the reach.

In [ ]:
def load_his(root, tref):
    ds = xr.open_dataset(root / 'sfincs_his.nc')
    raw = ds['station_name'].values
    names = []
    for row in raw:
        first = row[0]
        if isinstance(first, (bytes, np.bytes_)):
            names.append(b''.join(row).decode().strip())
        elif isinstance(first, (int, np.integer)):
            names.append(''.join(chr(c) for c in row if c != 0).strip())
        else:
            names.append(''.join(str(c) for c in row).strip())
    times = pd.DatetimeIndex(ds['time'].values)
    zs = ds['point_zs'].values   # (time, stations)
    zb = ds['point_zb'].values   # (stations,)
    depth = zs - zb[np.newaxis, :]
    return times, names, depth

times_sc, names_sc, depth_sc = load_his(model_root, tref)
times_bl, names_bl, depth_bl = load_his(baseline_root, tref)

mapping = pd.read_csv(mapping_file, sep=';')

t_start = pd.Timestamp('2021-07-12 00:00')
t_end   = pd.Timestamp('2021-07-17 00:00')

for i, sname in enumerate(names_sc):
    if sname not in ('Altenahr', 'Bad Bodendorf'):
        continue

    fig, ax = plt.subplots(figsize=(10, 4))

    mask_sc = (times_sc >= t_start) & (times_sc <= t_end)
    ax.plot(times_sc[mask_sc], depth_sc[mask_sc, i],
            label='Southonly dz5m', color='steelblue', linewidth=2)

    if i < depth_bl.shape[1]:
        mask_bl = (times_bl >= t_start) & (times_bl <= t_end)
        ax.plot(times_bl[mask_bl], depth_bl[mask_bl, i],
                label='Baseline (no infrastructure)', color='darkorange', linewidth=1.5, linestyle='--')

    row = mapping[mapping['name'] == sname]
    if not row.empty:
        obs_id = str(row['id'].iloc[0])
        obs_files = list(obs_dir.glob(f'{obs_id}_*.csv'))
        if obs_files:
            df_obs = pd.read_csv(obs_files[0], sep=';', decimal=',',
                                 parse_dates=['Datum'], dayfirst=True, encoding='latin1')
            df_obs = df_obs[(df_obs['Datum'] >= t_start) & (df_obs['Datum'] <= t_end)].copy()
            df_obs['Wert'] = df_obs['Wert'] / 100.0
            ax.plot(df_obs['Datum'], df_obs['Wert'], label='Observed', color='black', linewidth=1.5)

    ax.set_title(f'Station {i}: {sname}')
    ax.set_ylabel('Water depth [m]'); ax.set_xlabel('Time')
    ax.grid(True); ax.legend()
    plt.tight_layout(); plt.show()

## 3. Peak flood depth map, with the levee alignment overlaid

In [ ]:
def read_weir_lines(path):
    """Parse a sfincs.weir file into a list of (name, xy array) polylines."""
    lines = []
    with open(path) as f:
        content = f.read().split('\n')
    i = 0
    while i < len(content):
        if content[i].strip() == '':
            i += 1
            continue
        name = content[i].strip(); i += 1
        nrows, ncols = map(int, content[i].split()); i += 1
        pts = []
        for _ in range(nrows):
            parts = content[i].split()
            pts.append((float(parts[0]), float(parts[1])))
            i += 1
        lines.append((name, np.array(pts)))
    return lines

peak_h = da_h.max(dim='time', skipna=True).compute().values
peak_plot = np.where(peak_h > hmin, peak_h, np.nan)

wet_total = int(np.nansum(peak_h > hmin))
print(f'Wet cells (peak > {hmin} m): {wet_total}')
print(f'Peak depth max: {np.nanmax(peak_h):.2f} m')

weir_lines = read_weir_lines(model_root / 'sfincs.weir')
print(f'Weir polylines in this scenario: {[name for name, _ in weir_lines]}')

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.pcolormesh(X, Y, peak_plot, cmap='viridis', vmin=0, vmax=10, shading='nearest')
plt.colorbar(im, ax=ax, label='Peak water depth [m]')
for name, xy in weir_lines:
    ax.plot(xy[:, 0], xy[:, 1], color='red', linewidth=2, label=name)
ax.legend(fontsize=8)
ax.set_title('Peak water depth - southonly dz5m (levee alignment in red)')
ax.set_xlabel('Easting [m]'); ax.set_ylabel('Northing [m]')
ax.set_aspect('equal')
plt.tight_layout(); plt.show()

## 4. Peak velocity map

In [ ]:
if 'vmax' in ds_map.data_vars:
    vmax_arr = ds_map['vmax'].compute().values
    while vmax_arr.ndim > 2:
        vmax_arr = np.nanmax(vmax_arr, axis=0)
elif 'u' in ds_map.data_vars and 'v' in ds_map.data_vars:
    u = ds_map['u'].compute().values
    v = ds_map['v'].compute().values
    vmax_arr = np.nanmax(np.sqrt(u**2 + v**2), axis=0)
else:
    print('No velocity data found -- skipping.')
    vmax_arr = None

if vmax_arr is not None:
    vmax_plot = np.where((peak_h > hmin) & np.isfinite(vmax_arr), vmax_arr, np.nan)
    fig, ax = plt.subplots(figsize=(9, 7))
    im = ax.pcolormesh(X, Y, vmax_plot, cmap='plasma', vmin=0, vmax=3, shading='nearest')
    plt.colorbar(im, ax=ax, label='Peak velocity [m/s]')
    ax.set_title('Peak velocity - southonly dz5m')
    ax.set_xlabel('Easting [m]'); ax.set_ylabel('Northing [m]')
    ax.set_aspect('equal')
    plt.tight_layout(); plt.show()

## 5. Structure effect — windowed comparison against the no-infrastructure baseline

Compares peak water depth in this scenario against the baseline run (identical forcing/spinup,
no weir) inside the padded bbox of the Bad Neuenahr-Ahrweiler / Heimersheim reach
(`WIN_X`, `WIN_Y` defined above). A south-bank-only levee is expected to reduce depths on the
south side of the channel while leaving the unprotected north bank largely unchanged.

In [ ]:
ds_map_bl = xr.open_dataset(baseline_root / 'sfincs_map.nc', decode_times=False, mask_and_scale=True)
zs_bl = ds_map_bl['zs']; zb_bl = ds_map_bl['zb']
da_h_bl = (zs_bl - zb_bl).where(zb_bl > -9999)
peak_h_bl = da_h_bl.max(dim='time', skipna=True).compute().values

win_mask = (X >= WIN_X[0]) & (X <= WIN_X[1]) & (Y >= WIN_Y[0]) & (Y <= WIN_Y[1])

peak_scn_win = np.where(win_mask, peak_h, np.nan)
peak_bl_win  = np.where(win_mask, peak_h_bl, np.nan)
diff_win     = peak_scn_win - peak_bl_win   # negative = this scenario is shallower (levee helping)

wet_scn = int(np.nansum(peak_scn_win > hmin))
wet_bl  = int(np.nansum(peak_bl_win  > hmin))
cell_area = 100.0 * 100.0

print(f'Window wet cells -- baseline: {wet_bl}, southonly dz5m: {wet_scn} '
      f'({wet_scn - wet_bl:+d} cells, {(wet_scn - wet_bl) * cell_area / 1e4:+.1f} ha)')
print(f'Window max depth -- baseline: {np.nanmax(peak_bl_win):.2f} m, '
      f'southonly dz5m: {np.nanmax(peak_scn_win):.2f} m')
print(f'Window mean depth change (wet-in-either cells): {np.nanmean(diff_win):+.3f} m')

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True, sharey=True)

im0 = axes[0].pcolormesh(X, Y, np.where(peak_bl_win > hmin, peak_bl_win, np.nan),
                          cmap='viridis', vmin=0, vmax=10, shading='nearest')
axes[0].set_title('Baseline (no infrastructure)')
plt.colorbar(im0, ax=axes[0], label='Peak depth [m]', shrink=0.7)

im1 = axes[1].pcolormesh(X, Y, np.where(peak_scn_win > hmin, peak_scn_win, np.nan),
                          cmap='viridis', vmin=0, vmax=10, shading='nearest')
axes[1].set_title('Southonly dz5m')
for name, xy in weir_lines:
    axes[1].plot(xy[:, 0], xy[:, 1], color='red', linewidth=2)
plt.colorbar(im1, ax=axes[1], label='Peak depth [m]', shrink=0.7)

vabs = np.nanmax(np.abs(diff_win)) if np.any(np.isfinite(diff_win)) else 1.0
im2 = axes[2].pcolormesh(X, Y, diff_win, cmap='RdBu_r', vmin=-vabs, vmax=vabs, shading='nearest')
axes[2].set_title('Difference (scenario - baseline)\nblue = shallower with levee')
plt.colorbar(im2, ax=axes[2], label='Delta depth [m]', shrink=0.7)

for ax in axes:
    ax.set_xlim(WIN_X); ax.set_ylim(WIN_Y)
    ax.set_aspect('equal')
    ax.set_xlabel('Easting [m]')
axes[0].set_ylabel('Northing [m]')

plt.suptitle('Levee effect on peak flood depth - Bad Neuenahr-Ahrweiler / Heimersheim reach')
plt.tight_layout()
plt.show()

## 6. Flood reduction specifically in built-up (town) areas

Section 5 compares the whole reach window, which includes river channel, fields, and floodplain,
not just the town itself. To isolate the effect on the actual built-up area (Bad Neuenahr-Ahrweiler
/ Heimersheim), this section masks the window to "Built-up" cells (class 50) from the ESA
WorldCover 10m 2021 map already used for this model's roughness setup
(`data/esa_worldcover/.../ESA_WorldCover_10m_2021_v200_N48E006_Map.tif`), and repeats the
baseline-vs-scenario depth comparison only on those cells.

In [ ]:
import rasterio
from rasterio.warp import transform as warp_transform

worldcover_path = (project_root / 'data' / 'esa_worldcover' / 'WORLDCOVER' /
                    'ESA_WORLDCOVER_10M_2021_V200' / 'MAP' /
                    'ESA_WorldCover_10m_2021_v200_N48E006_Map' /
                    'ESA_WorldCover_10m_2021_v200_N48E006_Map.tif')
BUILTUP_CLASS = 50  # ESA WorldCover land-cover code for built-up / urban

lon_list, lat_list = warp_transform('EPSG:32632', 'EPSG:4326', X.ravel().tolist(), Y.ravel().tolist())
lon_flat, lat_flat = np.array(lon_list), np.array(lat_list)

with rasterio.open(worldcover_path) as src:
    rows, cols = rasterio.transform.rowcol(src.transform, lon_flat, lat_flat)
    rows = np.clip(np.array(rows), 0, src.height - 1)
    cols = np.clip(np.array(cols), 0, src.width - 1)
    # windowed read -- only the small patch covering this SFINCS domain, not the full 36000x36000 tile
    r0, r1 = rows.min(), rows.max()
    c0, c1 = cols.min(), cols.max()
    window = rasterio.windows.Window(int(c0), int(r0), int(c1 - c0 + 1), int(r1 - r0 + 1))
    wc_window = src.read(1, window=window)
    landcover = wc_window[rows - r0, cols - c0].reshape(X.shape)

builtup_mask = (landcover == BUILTUP_CLASS)
win_builtup_mask = win_mask & builtup_mask
print(f'Built-up SFINCS cells in the whole domain: {int(builtup_mask.sum())}')
print(f'Built-up SFINCS cells inside the evaluation window: {int(win_builtup_mask.sum())}')

peak_bl_win_bu  = np.where(win_builtup_mask, peak_h_bl, np.nan)
peak_scn_win_bu = np.where(win_builtup_mask, peak_h, np.nan)
diff_win_bu     = peak_scn_win_bu - peak_bl_win_bu

wet_bl_bu  = int(np.nansum(peak_bl_win_bu  > hmin))
wet_scn_bu = int(np.nansum(peak_scn_win_bu > hmin))

print(f'\nBuilt-up wet cells -- baseline: {wet_bl_bu}, southonly dz5m: {wet_scn_bu} '
      f'({wet_scn_bu - wet_bl_bu:+d} cells, {(wet_scn_bu - wet_bl_bu) * cell_area / 1e4:+.2f} ha)')
if wet_bl_bu > 0:
    print(f'Relative change in flooded built-up area: {(wet_scn_bu - wet_bl_bu) / wet_bl_bu:+.1%}')
print(f'Built-up max depth -- baseline: {np.nanmax(peak_bl_win_bu):.2f} m, '
      f'southonly dz5m: {np.nanmax(peak_scn_win_bu):.2f} m')
print(f'Built-up mean depth change (wet-in-either cells): {np.nanmean(diff_win_bu):+.3f} m')

fig, axes = plt.subplots(1, 2, figsize=(13, 6), sharex=True, sharey=True)

for ax, peak_arr, title in [(axes[0], peak_bl_win, 'Baseline (no infrastructure)'),
                             (axes[1], peak_scn_win, 'Southonly dz5m')]:
    im = ax.pcolormesh(X, Y, np.where(peak_arr > hmin, peak_arr, np.nan),
                        cmap='viridis', vmin=0, vmax=10, shading='nearest')
    # outline built-up cells
    ax.contour(X, Y, win_builtup_mask.astype(float), levels=[0.5], colors='red', linewidths=1.2)
    ax.set_title(title)
    ax.set_xlim(WIN_X); ax.set_ylim(WIN_Y)
    ax.set_aspect('equal')
    ax.set_xlabel('Easting [m]')
    plt.colorbar(im, ax=ax, label='Peak depth [m]', shrink=0.7)

axes[0].set_ylabel('Northing [m]')
plt.suptitle('Peak flood depth with built-up-area outline (red) -- Ahrweiler/Heimersheim')
plt.tight_layout()
plt.show()

## 7. Comparison across all infrastructure-placement scenarios

Repeats the same windowed / built-up-only comparison for all 14 generated scenarios (12 train +
2 held-out test, see `setup_infrastructure_placement_runs.py`), against the same baseline.

In [ ]:
INFRA_TAGS = ['bothbanks_dz2m', 'bothbanks_dz5m', 'bothbanks_dz10m','northonly_dz2m', 'northonly_dz5m',
              'southonly_dz2m', 'southonly_dz5m', 'upstream_dz2m', 'upstream_dz5m',
              'downstream_dz2m', 'downstream_dz5m', 'setback_dz2m', 'setback_dz5m',
              'gap_dz5m', 'reservoir']


def compute_peak_depth(root):
    ds = xr.open_dataset(root / 'sfincs_map.nc', decode_times=False, mask_and_scale=True)
    h = (ds['zs'] - ds['zb']).where(ds['zb'] > -9999)
    return h.max(dim='time', skipna=True).compute().values


summary_rows = [{
    'scenario': 'baseline (no infrastructure)',
    'window_wet_cells': wet_bl, 'window_wet_delta': 0,
    'window_max_depth_m': float(np.nanmax(peak_bl_win)),
    'builtup_wet_cells': wet_bl_bu, 'builtup_wet_delta': 0,
    'builtup_mean_depth_change_m': 0.0,
}]

# Stashed for the map visualisations in sections 8-9, so sfincs_map.nc isn't reopened/reduced again.
scenario_peak_win = {}
scenario_peak_full = {}
scenario_weir_lines = {}

for tag in INFRA_TAGS:
    root = project_root / 'models' / f'ahr_river_v03_Marg_additionalsrc_velocity_100m_cutpolygon_warmstart_infra_{tag}'
    if not root.exists():
        print(f'Missing folder for {tag}, skipping.')
        continue

    peak = compute_peak_depth(root)
    peak_win    = np.where(win_mask, peak, np.nan)
    peak_win_bu = np.where(win_builtup_mask, peak, np.nan)
    scenario_peak_win[tag] = peak_win
    scenario_peak_full[tag] = peak

    weir_path = root / 'sfincs.weir'
    scenario_weir_lines[tag] = read_weir_lines(weir_path) if weir_path.exists() else []

    w_wet    = int(np.nansum(peak_win > hmin))
    bu_wet   = int(np.nansum(peak_win_bu > hmin))
    bu_diff  = peak_win_bu - peak_bl_win_bu

    summary_rows.append({
        'scenario': tag,
        'window_wet_cells': w_wet, 'window_wet_delta': w_wet - wet_bl,
        'window_max_depth_m': float(np.nanmax(peak_win)) if np.any(np.isfinite(peak_win)) else np.nan,
        'builtup_wet_cells': bu_wet, 'builtup_wet_delta': bu_wet - wet_bl_bu,
        'builtup_mean_depth_change_m': float(np.nanmean(bu_diff)) if np.any(np.isfinite(bu_diff)) else np.nan,
    })

df_summary = pd.DataFrame(summary_rows).set_index('scenario', drop=False)
df_summary['builtup_wet_delta_ha'] = df_summary['builtup_wet_delta'] * cell_area / 1e4
df_summary_sorted = df_summary.sort_values('builtup_wet_delta')
df_summary_sorted

In [ ]:
# Bar chart: reduction in flooded built-up area vs baseline, most effective first
plot_df = df_summary_sorted[df_summary_sorted['scenario'] != 'baseline (no infrastructure)']

fig, ax = plt.subplots(figsize=(9, 6))
colors = ['seagreen' if v < 0 else 'firebrick' for v in plot_df['builtup_wet_delta']]
ax.barh(plot_df['scenario'], plot_df['builtup_wet_delta'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Change in flooded built-up cells vs baseline (negative = less flooding)')
ax.set_title('Built-up (town) area flood reduction by scenario\nAhrweiler/Heimersheim window')
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 7b. Diagnostic — do the weir lines fall inside/adjacent to river-channel cells?

Before generating more scenarios (setback distances, extra crest-height sweeps), check directly
whether the *current* placements have the suspected problem: SFINCS weir structures are snapped
onto edges of the 100 m flux grid, not smoothed into the terrain — the 10 m subgrid only refines
the sampled crest *height*, not which flux-grid edge the weir is structurally attached to. If a
digitized levee line at the immediate riverbank shares a flux cell with the channel itself, it
isn't acting like a floodplain-edge levee; it's partially throttling in-channel conveyance, which
would produce backwater/afflux and could explain *more* flooding rather than less.

**"Channel cell" proxy:** a cell that's already wet under baseflow-only conditions — the
warm-start water depth at t=0 in the `baseline` run, before the flood wave arrives (same
mechanism as the t=0 sanity check in Section 1, just from the baseline run so it's independent of
any scenario's own structure). For each scenario's weir polyline(s), the line is densely resampled
(20 m spacing, finer than the 100 m flux cell so no cell along the line is skipped) and every
sample point is checked against this channel mask, plus its distance to the nearest channel cell
(0 m / negative-equivalent = sitting directly on a channel cell).

In [ ]:
from scipy.spatial import cKDTree
from shapely.geometry import LineString

# Channel-cell mask: baseline t=0 (warm-start baseflow only, before the flood wave) wet cells.
ds_map_bl0 = xr.open_dataset(baseline_root / 'sfincs_map.nc', decode_times=False, mask_and_scale=True)
h0_bl = (ds_map_bl0['zs'].isel(time=0) - ds_map_bl0['zb']).where(ds_map_bl0['zb'] > -9999).values
channel_mask = h0_bl > hmin
n_rows_g, n_cols_g = channel_mask.shape
print(f'Channel cells (baseflow-wet at t=0, baseline): {int(channel_mask.sum())} of {channel_mask.size}')

channel_rows, channel_cols = np.where(channel_mask)
channel_xy = np.column_stack([
    x0_m + (channel_cols + 0.5) * dx,
    y0_m + (channel_rows + 0.5) * dy,
])
channel_tree = cKDTree(channel_xy)


def cell_index(x, y):
    col = np.floor((x - x0_m) / dx).astype(int)
    row = np.floor((y - y0_m) / dy).astype(int)
    return row, col


def diagnose_weir(lines, spacing=20.0):
    """Densely sample each weir polyline; check overlap/clearance against channel_mask."""
    n_pts, n_on_channel, min_clear = 0, 0, np.inf
    for _name, xy in lines:
        line = LineString(xy[:, :2])
        n_samples = max(2, int(line.length / spacing) + 1)
        pts = [line.interpolate(t) for t in np.linspace(0, line.length, n_samples)]
        xs = np.array([p.x for p in pts]); ys = np.array([p.y for p in pts])
        rows, cols = cell_index(xs, ys)
        in_bounds = (rows >= 0) & (rows < n_rows_g) & (cols >= 0) & (cols < n_cols_g)
        n_pts += len(xs)
        n_on_channel += int(channel_mask[rows[in_bounds], cols[in_bounds]].sum())
        dist, _ = channel_tree.query(np.column_stack([xs, ys]))
        min_clear = min(min_clear, float(dist.min()))
    return n_pts, n_on_channel, min_clear


rows_diag = []
for tag, lines in scenario_weir_lines.items():
    if not lines:
        continue
    n_pts, n_on_channel, min_clear = diagnose_weir(lines)
    rows_diag.append({
        'scenario': tag,
        'weir_sample_points': n_pts,
        'points_on_channel_cell': n_on_channel,
        'pct_on_channel': 100 * n_on_channel / n_pts,
        'min_clearance_to_channel_m': min_clear,
    })

df_diag = pd.DataFrame(rows_diag).set_index('scenario', drop=False).sort_values('min_clearance_to_channel_m')
df_diag

In [ ]:
# Visual check for this notebook's own scenario (southonly dz5m): channel cells (blue) vs the
# weir line (red), zoomed to the reach window. Overlap/near-touching red-on-blue is the failure
# mode this diagnostic is checking for.
fig, ax = plt.subplots(figsize=(9, 7))
ax.pcolormesh(X, Y, np.where(channel_mask, 1, np.nan), cmap='Blues', vmin=0, vmax=1.4, shading='nearest')
for name, xy in weir_lines:
    ax.plot(xy[:, 0], xy[:, 1], color='red', linewidth=2, label=name)
ax.set_xlim(WIN_X); ax.set_ylim(WIN_Y)
ax.set_aspect('equal')
ax.set_title('Channel cells (blue, baseflow-wet at t=0) vs southonly dz5m weir line (red)')
ax.set_xlabel('Easting [m]'); ax.set_ylabel('Northing [m]')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

this_diag = df_diag.loc['southonly_dz5m']
print(f"\nsouthonly_dz5m: {this_diag['points_on_channel_cell']}/{this_diag['weir_sample_points']} "
      f"sample points ({this_diag['pct_on_channel']:.1f}%) land on a channel cell; "
      f"min clearance to nearest channel cell = {this_diag['min_clearance_to_channel_m']:.1f} m.")

## 8. Flood-change maps for every scenario

Spatial view to go with the summary table above: peak-depth difference (scenario minus baseline)
for all 14 scenarios, same window and color scale throughout so they're directly comparable.
Blue = shallower/less flooded than baseline, red = deeper/more flooded. The black outline marks
built-up cells; the green line (where present) is the scenario's own weir alignment -- the
`reservoir` and `gap_dz5m` panels have no weir line drawn inside this window (reservoir sits at
Niederadenau, well outside it; gap's segments are drawn but the gap itself sits inside the window).

In [ ]:
tags_ordered = list(scenario_peak_win.keys())
ncols = 2   # the reach window is much wider than tall (~2.8:1) -- 2 columns keeps each panel readable
nrows = int(np.ceil(len(tags_ordered) / ncols))

# Size each panel from the window's own aspect ratio, so pcolormesh's aspect='equal' fills the panel
# instead of shrinking into a thin strip surrounded by white space (what a naive square figsize gives
# for a window this elongated).
win_w = WIN_X[1] - WIN_X[0]
win_h = WIN_Y[1] - WIN_Y[0]
panel_w = 7.0
panel_h = panel_w * win_h / win_w + 0.8   # +0.8in headroom for the two-line title

all_diffs = {tag: scenario_peak_win[tag] - peak_bl_win for tag in tags_ordered}
vabs = np.nanmax([np.nanmax(np.abs(d)) for d in all_diffs.values() if np.any(np.isfinite(d))])

fig, axes = plt.subplots(nrows, ncols, figsize=(panel_w * ncols, panel_h * nrows), sharex=True, sharey=True)
axes = np.array(axes).reshape(-1)

im = None
for ax, tag in zip(axes, tags_ordered):
    im = ax.pcolormesh(X, Y, all_diffs[tag], cmap='RdBu_r', vmin=-vabs, vmax=vabs, shading='nearest')
    ax.contour(X, Y, win_builtup_mask.astype(float), levels=[0.5], colors='black', linewidths=0.6)
    for name, xy in scenario_weir_lines[tag]:
        ax.plot(xy[:, 0], xy[:, 1], color='lime', linewidth=1.5)
    bu_delta = df_summary.loc[tag, 'builtup_wet_delta']
    ax.set_title(f'{tag}  (built-up: {bu_delta:+d} cells)', fontsize=10)
    ax.set_xlim(WIN_X); ax.set_ylim(WIN_Y)
    ax.set_aspect('equal')
    ax.tick_params(labelsize=7)

for ax in axes[len(tags_ordered):]:
    ax.axis('off')

fig.colorbar(im, ax=axes.tolist(), label='Delta peak depth [m] (scenario - baseline)', shrink=0.6, aspect=40)
plt.suptitle('Flood change vs baseline, all infrastructure scenarios', fontsize=13)
plt.show()

## 9. Full-catchment flood extent, before vs. after, for every scenario

Same peak-depth maps as everywhere else in this notebook, but zoomed out to the **whole SFINCS
domain** rather than just the reach window, so you can see the flood in its full catchment context
-- and confirm the infrastructure only changes things locally, not elsewhere in the domain. The red
dashed box marks the reach window used in sections 5-8. One figure per scenario (baseline on the
left, that scenario on the right), same color scale throughout.

In [ ]:
domain_w = X.max() - X.min()
domain_h = Y.max() - Y.min()
panel_w_full = 5.6
panel_h_full = panel_w_full * domain_h / domain_w + 0.6
vmax_full = 10.0  # same scale as sections 3, 5, 6, 8

win_box_x = [WIN_X[0], WIN_X[1], WIN_X[1], WIN_X[0], WIN_X[0]]
win_box_y = [WIN_Y[0], WIN_Y[0], WIN_Y[1], WIN_Y[1], WIN_Y[0]]

for tag in tags_ordered:
    fig, axes = plt.subplots(1, 2, figsize=(panel_w_full * 2, panel_h_full), sharex=True, sharey=True)

    for ax, arr, title in [(axes[0], peak_h_bl, 'Baseline (no infrastructure)'),
                            (axes[1], scenario_peak_full[tag], tag)]:
        im = ax.pcolormesh(X, Y, np.where(arr > hmin, arr, np.nan),
                            cmap='viridis', vmin=0, vmax=vmax_full, shading='nearest')
        ax.plot(win_box_x, win_box_y, color='red', linewidth=1.2, linestyle='--')
        ax.set_title(title)
        ax.set_aspect('equal')
        ax.set_xlabel('Easting [m]')

    axes[0].set_ylabel('Northing [m]')
    fig.colorbar(im, ax=axes.tolist(), label='Peak depth [m]', shrink=0.75)
    plt.suptitle(f'Full-catchment peak flood extent -- {tag}', fontsize=12)
    plt.show()
    plt.close(fig)

## 10. Flood evolution over time, all scenarios

Everything above only looked at the *peak* -- a single snapshot in time. This section tracks the
whole 121-hour hydrograph: flooded area (window and built-up-only) at every output timestep, for
baseline plus all 14 scenarios on the same axes, so you can see not just how much flooding each
scenario changes but also *when* -- earlier/later peak, faster/slower recession, etc.

In [ ]:
# Crop to the window's own row/col range before reducing over time -- much less data to read
# per scenario than loading the full domain x all 121 timesteps just to mask most of it away.
rows_win, cols_win = np.where(win_mask)
r0, r1 = rows_win.min(), rows_win.max()
c0, c1 = cols_win.min(), cols_win.max()
win_mask_crop = win_mask[r0:r1 + 1, c0:c1 + 1]
win_builtup_mask_crop = win_builtup_mask[r0:r1 + 1, c0:c1 + 1]


def compute_window_timeseries(root):
    ds = xr.open_dataset(root / 'sfincs_map.nc', decode_times=False, mask_and_scale=True)
    zs = ds['zs'].isel(n=slice(r0, r1 + 1), m=slice(c0, c1 + 1))
    zb = ds['zb'].isel(n=slice(r0, r1 + 1), m=slice(c0, c1 + 1))
    h = (zs - zb).where(zb > -9999)

    win_da = xr.DataArray(win_mask_crop, dims=('n', 'm'))
    bu_da  = xr.DataArray(win_builtup_mask_crop, dims=('n', 'm'))

    wet_win_ha = ((h.where(win_da) > hmin).sum(dim=('n', 'm')).astype(float) * cell_area / 1e4).values
    wet_bu_ha  = ((h.where(bu_da)  > hmin).sum(dim=('n', 'm')).astype(float) * cell_area / 1e4).values

    time_sec = ds['time'].values.astype(float)
    times = pd.to_datetime(time_sec, unit='s', origin=tref)
    return times, wet_win_ha, wet_bu_ha


run_list = [('baseline', baseline_root)] + [
    (tag, project_root / 'models' / f'ahr_river_v03_Marg_additionalsrc_velocity_100m_cutpolygon_warmstart_infra_{tag}')
    for tag in tags_ordered
]

evolution = {}
for label, root in run_list:
    evolution[label] = compute_window_timeseries(root)

print(f'Computed flood-evolution time series for {len(evolution)} runs '
      f'({len(evolution["baseline"][0])} timesteps each).')

In [ ]:
cmap = plt.get_cmap('tab20')
color_map = {tag: cmap((i % 20) / 19) for i, tag in enumerate(tags_ordered)}

fig, axes = plt.subplots(2, 1, figsize=(11, 9), sharex=True)

for label, (times, wet_win_ha, wet_bu_ha) in evolution.items():
    if label == 'baseline':
        kwargs = dict(color='black', linewidth=2.5, linestyle='-', zorder=10, label='baseline')
    else:
        kwargs = dict(color=color_map[label], linewidth=1.3,
                       linestyle=('--' if 'dz2m' in label else '-'), label=label)
    axes[0].plot(times, wet_win_ha, **kwargs)
    axes[1].plot(times, wet_bu_ha, **kwargs)

axes[0].set_ylabel('Flooded area [ha]')
axes[0].set_title('Whole reach window')
axes[1].set_ylabel('Flooded area [ha]')
axes[1].set_title('Built-up (town) cells only')
axes[1].set_xlabel('Time')

for ax in axes:
    ax.grid(True, alpha=0.3)

axes[0].legend(loc='upper left', bbox_to_anchor=(1.01, 1.0), fontsize=8, ncol=1,
               title='dashed = dz2m, solid = dz5m')
plt.suptitle('Flood evolution over time -- baseline vs all infrastructure scenarios', fontsize=13)
plt.tight_layout()
plt.show()

## 11. Flood extent at 4 key moments, scenario vs baseline

The peak-only comparisons above (sections 5-9) miss the flood's *shape* over time. This section
picks 4 characteristic moments straight off the baseline hydrograph computed in section 10 --
**rising limb** (last point before the peak still below half the peak's flooded area), **peak**,
**falling limb** (first point after the peak back below half the peak value, if reached), and
**late** (the last common timestep -- baseline is the shorter series at t=118h, see the truncation
note above) -- and shows the window-cropped depth map for baseline vs each scenario side by side at
each moment. Note the print output below: if recession is slow enough that flooded area never
drops back below half its peak within the series, "falling"/"late" fall back to fixed offsets from
the peak instead -- worth checking, since it changes what those two labels actually represent.

In [ ]:
times_bl, wet_win_bl, wet_bu_bl = evolution['baseline']
peak_idx = int(np.argmax(wet_win_bl))
peak_val = wet_win_bl[peak_idx]

rising_candidates = np.where(wet_win_bl[:peak_idx] <= 0.5 * peak_val)[0]
rising_idx = int(rising_candidates[-1]) if len(rising_candidates) else max(0, peak_idx - 10)

falling_candidates = np.where(wet_win_bl[peak_idx:] <= 0.5 * peak_val)[0]
falling_idx = int(peak_idx + falling_candidates[0]) if len(falling_candidates) else min(len(wet_win_bl) - 1, peak_idx + 10)

late_idx = len(wet_win_bl) - 1  # last common timestep (baseline is the shorter/truncated series)

key_indices = {'rising': rising_idx, 'peak': peak_idx, 'falling': falling_idx, 'late': late_idx}
for label, idx in key_indices.items():
    elapsed_h = (times_bl[idx] - tref).total_seconds() / 3600
    print(f'{label:8s}: index {idx:3d}, t={times_bl[idx]}  (t+{elapsed_h:.0f}h), '
          f'window flooded area = {wet_win_bl[idx]:.1f} ha')

X_crop = X[r0:r1 + 1, c0:c1 + 1]
Y_crop = Y[r0:r1 + 1, c0:c1 + 1]


def load_window_snapshots(root, indices):
    ds = xr.open_dataset(root / 'sfincs_map.nc', decode_times=False, mask_and_scale=True)
    zs = ds['zs'].isel(n=slice(r0, r1 + 1), m=slice(c0, c1 + 1), time=indices)
    zb = ds['zb'].isel(n=slice(r0, r1 + 1), m=slice(c0, c1 + 1))
    h = (zs - zb).where(zb > -9999)
    return h.values  # shape (len(indices), rows, cols)


baseline_snaps = load_window_snapshots(baseline_root, list(key_indices.values()))
print(f'\nBaseline snapshot array shape: {baseline_snaps.shape}')

In [ ]:
key_labels = list(key_indices.keys())

for tag in tags_ordered:
    root = project_root / 'models' / f'ahr_river_v03_Marg_additionalsrc_velocity_100m_cutpolygon_warmstart_infra_{tag}'
    scn_snaps = load_window_snapshots(root, list(key_indices.values()))

    fig, axes = plt.subplots(2, 4, figsize=(4.6 * 4, panel_h * 2), sharex=True, sharey=True)

    im = None
    for j, label in enumerate(key_labels):
        elapsed_h = (times_bl[key_indices[label]] - tref).total_seconds() / 3600

        im = axes[0, j].pcolormesh(X_crop, Y_crop,
                                    np.where(baseline_snaps[j] > hmin, baseline_snaps[j], np.nan),
                                    cmap='viridis', vmin=0, vmax=vmax_full, shading='nearest')
        axes[0, j].set_title(f'{label} (t+{elapsed_h:.0f}h)\nbaseline', fontsize=9)

        axes[1, j].pcolormesh(X_crop, Y_crop,
                               np.where(scn_snaps[j] > hmin, scn_snaps[j], np.nan),
                               cmap='viridis', vmin=0, vmax=vmax_full, shading='nearest')
        axes[1, j].set_title(tag, fontsize=9)
        for name, xy in scenario_weir_lines[tag]:
            axes[1, j].plot(xy[:, 0], xy[:, 1], color='red', linewidth=1.2)

    for ax in axes.ravel():
        ax.set_xlim(WIN_X); ax.set_ylim(WIN_Y)
        ax.set_aspect('equal')
        ax.tick_params(labelsize=6)

    fig.colorbar(im, ax=axes.ravel().tolist(), label='Depth [m]', shrink=0.7)
    plt.suptitle(f'Flood extent at 4 key moments -- baseline (top) vs {tag} (bottom)', fontsize=13)
    plt.show()
    plt.close(fig)

## 12. Altenahr bridge-blockage scenarios — different site, different mechanism

Everything above is the Ahrweiler/Heimersheim levee placement study (`INFRA_TAGS`). This section
is a separate experiment at the Altenahr gorge, upstream of that reach, representing a
**debris-clogged bridge** — a failure mode that actually occurred in the real 2021 event (several
bridges got jammed with debris and acted as temporary dams, causing local backwater before
eventually failing).

Two candidate crossing locations (`data/measures/thin_dam.geojson`, `thin_dam2.geojson`), each
drawn straight across the channel rather than along the bank. First attempt used a true `thd`
(infinite-height, zero-tolerance block) and it broke the solver — an isolated cell hit 47 m depth
with a NaN gap in its time series right before the run-away, while immediate neighbors at similar
bed elevation stayed dry (not a real, spatially coherent backwater pool). Rebuilt as a **weir with
`dz=15m`** instead — keeps the overtopping formula active (no hard zero/infinity discontinuity)
while still being high enough to be practically un-overtopped for this event. Result was stable: a
smooth, gradually-building rise with no NaNs (see conversation), telling the same "severe channel
constriction" story without the numerical blow-up.

Own evaluation window here (`WIN_X2`, `WIN_Y2`) since this site sits upstream of, and does not
overlap with, the Ahrweiler reach window used everywhere above.

In [ ]:
WIN_X2 = (353000.0, 362500.0)
WIN_Y2 = (5594500.0, 5600000.0)
win_mask2 = (X >= WIN_X2[0]) & (X <= WIN_X2[1]) & (Y >= WIN_Y2[0]) & (Y <= WIN_Y2[1])

ALTENAHR_TAGS = ['thin_dam_dz15m', 'thin_dam_dz20m', 'thin_dam2_dz15m', 'thin_dam2_dz20m']

peak_bl_win2 = np.where(win_mask2, peak_h_bl, np.nan)
wet_bl2 = int(np.nansum(peak_bl_win2 > hmin))

altenahr_summary = []
for tag in ALTENAHR_TAGS:
    root = project_root / 'models' / f'ahr_river_v03_Marg_additionalsrc_velocity_100m_cutpolygon_warmstart_infra_{tag}'
    peak = compute_peak_depth(root)
    peak_win2 = np.where(win_mask2, peak, np.nan)
    diff2 = peak_win2 - peak_bl_win2
    wet2 = int(np.nansum(peak_win2 > hmin))

    max_inc = float(np.nanmax(diff2)); max_dec = float(np.nanmin(diff2))
    weir_path = root / 'sfincs.weir'
    weir_lines2 = read_weir_lines(weir_path) if weir_path.exists() else []

    altenahr_summary.append({
        'scenario': tag, 'window_wet_cells': wet2, 'window_wet_delta': wet2 - wet_bl2,
        'max_depth_increase_m': max_inc, 'max_depth_decrease_m': max_dec,
    })

    fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True, sharey=True)
    im0 = axes[0].pcolormesh(X, Y, np.where(peak_bl_win2 > hmin, peak_bl_win2, np.nan),
                              cmap='viridis', vmin=0, vmax=26, shading='nearest')
    axes[0].set_title('Baseline (no blockage)')
    plt.colorbar(im0, ax=axes[0], label='Peak depth [m]', shrink=0.7)

    im1 = axes[1].pcolormesh(X, Y, np.where(peak_win2 > hmin, peak_win2, np.nan),
                              cmap='viridis', vmin=0, vmax=26, shading='nearest')
    axes[1].set_title(tag)
    for name, xy in weir_lines2:
        axes[1].plot(xy[:, 0], xy[:, 1], color='red', linewidth=2)
    plt.colorbar(im1, ax=axes[1], label='Peak depth [m]', shrink=0.7)

    vabs2 = np.nanmax(np.abs(diff2)) if np.any(np.isfinite(diff2)) else 1.0
    im2 = axes[2].pcolormesh(X, Y, diff2, cmap='RdBu_r', vmin=-vabs2, vmax=vabs2, shading='nearest')
    axes[2].set_title('Difference (scenario - baseline)\nred = backwater rise')
    plt.colorbar(im2, ax=axes[2], label='Delta depth [m]', shrink=0.7)

    for ax in axes:
        ax.set_xlim(WIN_X2); ax.set_ylim(WIN_Y2)
        ax.set_aspect('equal')
        ax.set_xlabel('Easting [m]')
    axes[0].set_ylabel('Northing [m]')
    plt.suptitle(f'Bridge-blockage backwater effect — {tag} (Altenahr)')
    plt.tight_layout()
    plt.show()

df_altenahr = pd.DataFrame(altenahr_summary).set_index('scenario', drop=False)
print(f'Baseline window wet cells (Altenahr window): {wet_bl2}\n')
df_altenahr